<a href="https://colab.research.google.com/github/ansonkwokth/TableTennisPrediction/blob/dev/NNs_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/ansonkwokth/TableTennisPrediction.git
%cd TableTennisPrediction

Cloning into 'TableTennisPrediction'...
remote: Enumerating objects: 373, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 373 (delta 36), reused 15 (delta 15), pack-reused 311 (from 2)
Receiving objects: 100% (373/373), 6.79 MiB | 11.82 MiB/s, done.
Resolving deltas: 100% (182/182), done.
/content/TableTennisPrediction


In [3]:
import pandas as pd
from utils import data_loader as dl

import numpy as np
from model.Elo import Elo
from model.ModifiedElo import ModifiedElo
from model.ensemble import BaggingRatingSystem

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import copy
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Data

In [4]:
# GAME = 'TTStar'
# GAME = 'TTCup'

# GAME = 'SetkaCup'
GAME = 'SetkaCupWomen'
# GAME = 'LigaPro'


In [5]:
match GAME:
    case 'TTStar':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'TTCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCupWomen':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'LigaPro':
        years = [2022, 2023, 2024]
    case _:
        raise ValueError("Invalid game selected.")


text_data_game = dl.load_game_data(GAME, years, '../')
text_data = {
    year: text_data_game[year] for year in years
}
df = dl.create_game_dfs(GAME, years, text_data)

Loading ..//SetkaCupWomen2020.txt
Loading ..//SetkaCupWomen2021.txt
Loading ..//SetkaCupWomen2022.txt
Loading ..//SetkaCupWomen2023.txt
Loading ..//SetkaCupWomen2024.txt


In [6]:
# Generate ID indices for each pair of rows in the DataFrame
idx_lt = [i for i in range(len(df) // 2) for _ in range(2)]
df['ID'] = idx_lt  # Assign to the 'ID' column

# Reset the DataFrame index to ensure it's sequential
df.reset_index(drop=True, inplace=True)

# Get unique players and store them in player_lt
player_lt = df['Player'].unique()



In [7]:
year_val = years[-2]
year_test = years[-1]

df_train = df.loc[pd.DatetimeIndex(df['Date']).year < year_val]
df_val = df.loc[pd.DatetimeIndex(df['Date']).year == year_val]
df_train_val = df.loc[pd.DatetimeIndex(df['Date']).year <= year_val]
df_test = df.loc[pd.DatetimeIndex(df['Date']).year == year_test]

In [8]:
def format_to_array(df: pd.DataFrame) -> np.ndarray:

    # info_col = ['ID', 'Round', 'Datetime', 'Game', 'Date', 'Time']
    info_col = ['Round', 'Datetime', 'Game', 'Date', 'Time']
    col = [item for item in df.columns if item not in info_col]

    df[[c for c in col if "Set" in c]] = df[[c for c in col if "Set" in c]].astype(float)
    X = df[col].values.reshape(-1, 2, len(col))
    return X

In [9]:
X_train = format_to_array(df_train)
X_train_val = format_to_array(df_train_val)
X_val = format_to_array(df_val)
X_test = format_to_array(df_test)

In [10]:
X_all = format_to_array(df)

In [109]:

player_to_idx_train = {pn: i for i, pn in enumerate(np.unique(np.array(X_train)[:, :, 1].reshape(-1)))}
player_to_idx_train_val = {pn: i for i, pn in enumerate(np.unique(np.array(X_train_val)[:, :, 1].reshape(-1)))}


In [110]:
def get_data(X, player_to_idx):
    data_all = []
    for game in X:
        game = game[:, 1:]

        player1, player2 = game[:, 0]
        scores = game[:, 1:].astype(float)

        win1 = sum(scores[0]>scores[1])
        win2 = sum(scores[0]<scores[1])
        p1_win = int(win1 > win2)

        t = scores[0] / (scores[0] + scores[1])

        mask = (1-np.isnan(t)).astype(int)
        t = np.nan_to_num(t, nan=-1)

        datai = [player_to_idx[player1], player_to_idx[player2],
                 t, mask, p1_win]

        data_all.append(datai)
    return data_all

In [111]:
data_train_idx = get_data(X_train, player_to_idx_train)
data_train_val_idx = get_data(X_train, player_to_idx_train_val)

In [112]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader



class TableTennisDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        p1, p2, t, mask, p1_win = self.data[idx]
        return int(p1), int(p2), t, mask, p1_win


dataset_train = TableTennisDataset(data_train_idx)
dataloader_train = DataLoader(dataset_train, batch_size=64, shuffle=True)


dataset_train_val = TableTennisDataset(data_train_val_idx)
dataloader_train_val = DataLoader(dataset_train_val, batch_size=32, shuffle=True)



In [116]:
class SiameseNetwork(nn.Module):
    def __init__(self, num_players, embedding_dim=8):
        super(SiameseNetwork, self).__init__()
        # Shared embedding layer for players.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # A small MLP to convert the embedding into a scalar "score."
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 4),
            nn.Linear(4, 4),
            nn.Linear(4, 1)
        )

    def forward(self, player1_idx, player2_idx):
        # Process both players using the same embedding and MLP.
        embed1 = self.embedding(player1_idx)
        embed2 = self.embedding(player2_idx)
        score1 = self.fc(embed1).squeeze(-1)
        score2 = self.fc(embed2).squeeze(-1)
        # Directly return the win probability for player1.
        prob = 1.0 / (1.0 + torch.exp(score2 - score1))
        return prob


In [117]:
def loss_fn(p, t, mask):
    epsilon = 1e-7
    p = torch.clamp(p, epsilon, 1 - epsilon)
    tsum = torch.sum(t * mask, axis=1)

    loss = - (tsum * torch.log(p) + (1 - tsum) * torch.log(1 - p))

    return loss.mean()


In [118]:
num_players = len(player_to_idx_train)
model = SiameseNetwork(num_players, embedding_dim=8)
optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 50
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader_train:
        player1_idx, player2_idx, t, mask, p1_win = batch
        player1_idx = player1_idx.long()
        player2_idx = player2_idx.long()
        t = t.float()

        p = model(player1_idx, player2_idx)
        loss = loss_fn(p, t, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader_train):.4f}")



Epoch 1/50, Loss: 0.6747
Epoch 2/50, Loss: 0.6336
Epoch 3/50, Loss: 0.5831
Epoch 4/50, Loss: 0.5364
Epoch 5/50, Loss: 0.4896
Epoch 6/50, Loss: 0.4469
Epoch 7/50, Loss: 0.3981
Epoch 8/50, Loss: 0.3640
Epoch 9/50, Loss: 0.3302
Epoch 10/50, Loss: 0.3083
Epoch 11/50, Loss: 0.2563
Epoch 12/50, Loss: 0.2372
Epoch 13/50, Loss: 0.2070
Epoch 14/50, Loss: 0.1950
Epoch 15/50, Loss: 0.1673
Epoch 16/50, Loss: 0.1597
Epoch 17/50, Loss: 0.1581
Epoch 18/50, Loss: 0.1262
Epoch 19/50, Loss: 0.1169
Epoch 20/50, Loss: 0.1144
Epoch 21/50, Loss: 0.1016
Epoch 22/50, Loss: 0.0924
Epoch 23/50, Loss: 0.0890
Epoch 24/50, Loss: 0.0887
Epoch 25/50, Loss: 0.0672
Epoch 26/50, Loss: 0.0618
Epoch 27/50, Loss: 0.0663
Epoch 28/50, Loss: 0.0764
Epoch 29/50, Loss: 0.0642
Epoch 30/50, Loss: 0.0414
Epoch 31/50, Loss: 0.0436
Epoch 32/50, Loss: 0.0508
Epoch 33/50, Loss: 0.0473
Epoch 34/50, Loss: 0.0538
Epoch 35/50, Loss: 0.0313
Epoch 36/50, Loss: 0.0626
Epoch 37/50, Loss: 0.0267
Epoch 38/50, Loss: 0.0279
Epoch 39/50, Loss: 0.

In [120]:

model.eval()  # set model to evaluation mode
correct = 0
total = 0

trained_players = np.unique(X_train[:, :, 1])

with torch.no_grad():
    for batch in X_val:

        player1, player2 = batch[:, 1]
        if player1 not in trained_players: continue
        if player2 not in trained_players: continue

        player1_idx, player2_idx = player_to_idx_train[player1], player_to_idx_train[player2]

        player1_idx = torch.tensor([player1_idx]).long()
        player2_idx = torch.tensor([player2_idx]).long()
        win1 = (sum(batch[0, 2:]>batch[1, 2:]))
        win2 = (sum(batch[0, 2:]<batch[1, 2:]))


        # Ground truth: 1 if player1 won the set, 0 otherwise.
        ground_truth = (win1 > win2)
        # Predicted probability from the model.

        p = model(player1_idx, player2_idx)
        prediction = (p > 0.5).float()  # threshold at 0.5

        correct += int(prediction == ground_truth)
        total += 1

accuracy = correct / total
print("Test accuracy: {:.2f}%".format(accuracy * 100))

Test accuracy: 56.00%
